# EVAL — Dispersion P@5 et popularité

Ce notebook lit les trois bundles produits ensemble par `scripts/Models/evaluate_final.py` :

- `results/eval_baseline.json` ;
- `results/eval_cluster.json` ;
- `results/eval_cf.json`.

Ils partagent obligatoirement le même split test et les mêmes cas produit. L’écart-type présenté mesure la dispersion entre cas utilisateurs sur ce test unique, pas la robustesse entre plusieurs splits. La popularité est un diagnostic des recommandations, pas une feature de score.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

_ROOT = next(
    p for p in [Path("."), Path("..")]
    if (p / "scripts" / "Models" / "eval_diagnostics.py").exists()
)
sys.path.insert(0, str((_ROOT / "scripts" / "Models").resolve()))

from eval_diagnostics import (
    find_results_dir,
    flatten_methods,
    load_eval_bundles,
    plot_p5_variance,
    plot_popularity,
    popularity_table,
    shared_film_n_ratings,
    variance_table,
)

RESULTS = find_results_dir()
bundles = load_eval_bundles(RESULTS)
print(f"Bundles trouvés dans {RESULTS} : {len(bundles)}")
for b in bundles:
    print(
        f"  - {b.get('source')}: {len(b.get('methods', {}))} méthodes | "
        f"n_cases={b.get('n_cases')} | seed={b.get('random_state')} | "
        f"TRAIN_FRAC={b.get('train_frac')}"
    )

if not bundles:
    raise FileNotFoundError(
        "Aucun results/eval_*.json — lance d'abord MODEL_BASELINE, MODEL_CF, MODEL_CLUSTER."
    )

seeds = {b.get("random_state") for b in bundles}
fracs = {b.get("train_frac") for b in bundles}
if len(seeds) > 1 or len(fracs) > 1:
    print(
        "\nAttention : seeds ou TRAIN_FRAC différents entre bundles — "
        "la comparaison reste indicative."
    )


## 1. Variance des P@5


In [ ]:
methods = flatten_methods(bundles)
var_df = variance_table(methods)
display(var_df.round(4))

baseline_name = next(
    (m for m in var_df["method"] if m.endswith("/baseline_overlap_mean")),
    None,
)

plot_p5_variance(var_df, baseline_method=baseline_name)
plt.show()

print(
    "Écart-type élevé = P@5 très variable selon le profil / les seeds. "
    "Comparer surtout le P@5 moyen ; l'écart-type qualifie la stabilité."
)


## 2. Biais de popularité

Sur les **top-5** exportés : part des films dans le top-20 / top-100 du catalogue train (par volume de notes).


In [ ]:
film_n = shared_film_n_ratings(bundles)
if film_n.empty:
    raise ValueError("film_n_ratings manquant dans les bundles")

bucket_df = popularity_table(methods, film_n)
display(bucket_df.round(3))

plot_popularity(bucket_df)
plt.show()

print(
    "Plus la barre orange (top 20) est haute, plus le modèle recycle les films "
    "déjà très notés. Utile pour nuancer un bon P@5 : personnalisation vs popularité."
)


## Lecture courte

- Un modèle avec **P@5 haut** et **forte part top-20** personnalise peu : il suit le catalogue populaire.
- Un modèle avec **P@5 correct** et plus de **hors top 100** explore davantage (souvent FilmCF / contenu).
- La **variance** indique si le score tient sur tous les profils ou seulement sur une partie des cas.
